In [44]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.ml.feature import VectorAssembler, OneHotEncoder, StringIndexer
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
import os

In [2]:
spark: SparkSession = SparkSession.builder.appName("airbnb1").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/21 14:54:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
file_path = "/app/src/main/data/sf_airbnb/sf-airbnb-clean.parquet/"
os.path.exists(file_path)

airbnbdf = spark.read.parquet(file_path)
airbnbdf.select(
    "neighbourhood_cleansed",
    "room_type",
    "bedrooms",
    "bathrooms",
    "number_of_reviews",
    f.log("price").alias("price"),
).show(5)

+----------------------+---------------+--------+---------+-----------------+-----------------+
|neighbourhood_cleansed|      room_type|bedrooms|bathrooms|number_of_reviews|            price|
+----------------------+---------------+--------+---------+-----------------+-----------------+
|      Western Addition|Entire home/apt|     1.0|      1.0|            180.0|5.135798437050262|
|        Bernal Heights|Entire home/apt|     2.0|      1.0|            111.0|5.459585514144159|
|        Haight Ashbury|   Private room|     1.0|      4.0|             17.0|4.174387269895637|
|        Haight Ashbury|   Private room|     1.0|      4.0|              8.0|4.174387269895637|
|      Western Addition|Entire home/apt|     2.0|      1.5|             27.0|6.665683717782408|
+----------------------+---------------+--------+---------+-----------------+-----------------+
only showing top 5 rows


In [52]:
traindf, testdf = airbnbdf.randomSplit([0.8, 0.2], seed=42)
print(
    f"There are {traindf.count()} rows in the training set and {testdf.count()} rows in the test set"
)

There are 5780 rows in the training set and 1366 rows in the test set


In [53]:
# caching traindf
traindf.cache()
traindf.count()

5780

In [16]:
vec_assembler = VectorAssembler(inputCols=["bedrooms"], outputCol="features")
vec_traindf = vec_assembler.transform(traindf)
vec_traindf.select("bedrooms", "features", "price").show(5)

+--------+--------+-----+
|bedrooms|features|price|
+--------+--------+-----+
|     1.0|   [1.0]|200.0|
|     1.0|   [1.0]|130.0|
|     1.0|   [1.0]| 95.0|
|     1.0|   [1.0]|250.0|
|     3.0|   [3.0]|250.0|
+--------+--------+-----+
only showing top 5 rows


In [19]:
lr = LinearRegression(featuresCol="features", labelCol="price")
lr_model = lr.fit(vec_traindf)

In [20]:
m = round(lr_model.coefficients[0], 2)
b = round(lr_model.intercept, 2)
print(f"The formula for the linear regression line is price = {m}*bedrooms + {b}")

The formula for the linear regression line is price = 123.68*bedrooms + 47.51


In [22]:
# creating pipeline
pipeline = Pipeline(stages=[vec_assembler, lr])
pipeline_model = pipeline.fit(traindf)

In [26]:
pred_df = pipeline_model.transform(testdf)
pred_df.select("bedrooms", "features", "price", f.round(f.col("prediction"))).show(5)

+--------+--------+-----+--------------------+
|bedrooms|features|price|round(prediction, 0)|
+--------+--------+-----+--------------------+
|     1.0|   [1.0]| 85.0|               171.0|
|     1.0|   [1.0]| 45.0|               171.0|
|     1.0|   [1.0]| 70.0|               171.0|
|     1.0|   [1.0]|128.0|               171.0|
|     1.0|   [1.0]|159.0|               171.0|
+--------+--------+-----+--------------------+
only showing top 5 rows


In [54]:
categorical_cols = [
    field for (field, datatype) in traindf.dtypes if datatype == "string"
]
index_output_cols = [x + "Index" for x in categorical_cols]
ohe_output_cols = [x + "OHE" for x in categorical_cols]

string_indexer = StringIndexer(
    inputCols=categorical_cols, outputCols=index_output_cols, handleInvalid="skip"
)
ohe_encoder = OneHotEncoder(inputCols=index_output_cols, outputCols=ohe_output_cols)

numerical_cols = [
    field
    for (field, dtype) in traindf.dtypes
    if (dtype == "double") & (field != "price")
]

assembler_inputs = ohe_output_cols + numerical_cols
vec_assembler_multi = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

lr = LinearRegression(labelCol="price", featuresCol="features")
pipeline = Pipeline(stages=[string_indexer, ohe_encoder, vec_assembler_multi, lr])

pipeline_model_multi = pipeline.fit(traindf)
predf_multi = pipeline_model_multi.transform(testdf)
predf_multi.select("features", "price", "prediction").show(5)

+--------------------+-----+------------------+
|            features|price|        prediction|
+--------------------+-----+------------------+
|(98,[0,3,6,22,43,...| 85.0| 55.24365707389188|
|(98,[0,3,6,22,43,...| 45.0|23.357685914717877|
|(98,[0,3,6,22,43,...| 70.0|28.474464479034395|
|(98,[0,3,6,12,42,...|128.0| -91.6079079594947|
|(98,[0,3,6,12,43,...|159.0| 95.05688229945372|
+--------------------+-----+------------------+
only showing top 5 rows


In [ ]:
# evaluations
regression_evaluator = RegressionEvaluator(
    predictionCol="prediction", labelCol="price", metricName="rmse"
)

rmse = regression_evaluator.evaluate(predf_multi)
print(f"RMSE is {rmse:.1f}")

r2 = regression_evaluator.setMetricName("r2").evaluate(pred_df)
print(f"R2 is {r2}")

RMSE is 220.6
R2 is 0.15171845547373952
